In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import desc
from pyspark.sql.functions import asc
from pyspark.sql.functions import sum as Fsum

import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# download from google storage
import requests

url = 'https://storage.googleapis.com/files-bb-bot/music_log.json'
r = requests.get(url)
with open('./data/music_log.json', 'wb') as f:
    f.write(r.content)

In [ ]:
spark = SparkSession \
    .builder \
    .appName("fhtw music analytics") \
    .getOrCreate()

In [ ]:
df = spark.read.json("./data/music_log.json")

In [ ]:
df

### 1. RDD Query
**Description**: 
**Resilient Distributed Datasets (RDDs)** are the fundamental building blocks of Spark, providing a low-level, fine-grained data processing framework. RDDs are immutable, distributed collections of objects, which means once you create an RDD, you cannot change it. Each RDD is split into multiple partitions, which may be processed in parallel across different nodes of a Spark cluster.

**Use Cases**:
- **Complex Data Processing**: When the data processing task involves complex manipulations that are not supported directly by DataFrame or SQL operations.
- **Custom Partitioning**: RDDs allow you to control how data is distributed across the cluster, which can optimize specific types of operations.
- **Legacy Integration**: In scenarios where you need to integrate with legacy systems or algorithms originally designed for RDDs.

**Advantages**:
- **Flexibility**: Offers complete control over data processing and transformation steps.
- **Fine-grained Operations**: Supports detailed manipulation and processing of data at the element level.

In [ ]:
songs_rdd = df.rdd

In [ ]:
songs_rdd

 **Example Query**:

Assuming you have already loaded your RDD from the music log JSON file
Query to count song plays by each user using RDD:

In [ ]:
# Filter to include only song plays, then map and reduce by key
song_plays_by_user_rdd = songs_rdd \
    .filter(lambda x: x.page == 'NextSong') \
    .map(lambda x: (x.userId, 1)) \
    .reduceByKey(lambda x, y: x + y)

# Collect and print the result
collected_song_plays_rdd = song_plays_by_user_rdd.collect()
for user, count in collected_song_plays_rdd:
    print(f"User ID {user} has played {count} songs.")

### 2. DataFrame Query
**Description**:
**DataFrames** are a more modern abstraction in Spark, designed to simplify working with structured and semi-structured data. Like RDDs, DataFrames are immutable and distributed, but they are organized into named columns, much like a table in a relational database. This allows Spark to apply advanced optimizations based on the schema.

**Use Cases**:
- **Interactive Data Analysis**: Ideal for ad-hoc queries and exploratory data analysis because of their simplicity and built-in functions.
- **Machine Learning Pipelines**: DataFrames integrate seamlessly with MLlib, Spark's machine learning library, to preprocess data and train models.
- **Stream Processing**: Easily integrates with Structured Streaming to process real-time data streams.

**Advantages**:
- **Performance**: Catalyst optimizer improves performance by optimizing execution plans.
- **Ease of Use**: High-level API for complex transformations and aggregations.
- **Integration**: Supports a wide range of data sources and formats for input and output.

**Example Query**:


In [ ]:
from pyspark.sql.functions import col

# Assuming 'df' is the DataFrame loaded from the music log JSON file
# Count song plays by each user using DataFrame API
song_plays_by_user_df = df \
    .filter(col('page') == 'NextSong') \
    .groupBy('userId') \
    .count() \
    .withColumnRenamed('count', 'song_plays')

song_plays_by_user_df.show()

### 3. Spark SQL Query
**Description**:
**Spark SQL** is a module in Spark that allows for SQL and HiveQL querying syntax. It provides seamless integration between relational and procedural data manipulations, working across both DataFrames and Datasets. This module not only supports traditional SQL queries but also enables developers to intermix SQL queries with the programmatic data manipulations of DataFrames.

**Use Cases**:
- **SQL and Hive Users**: Offers a familiar interface for SQL and Hive users to run queries on big data.
- **Data Warehousing**: Can be used as part of a larger data warehousing setup that integrates with existing Hive setups.
- **Unified Data Access**: Query data directly from multiple sources, including JSON, Hive, Avro, Parquet, and ORC.

**Advantages**:
- **Optimization**: Benefits from Catalyst optimizer and Tungsten execution engine for efficient query execution.
- **Unified API**: Allows mixing SQL with functional programming, making complex workflows easier to develop and debug.

**Example Query**:

In [ ]:
# Register the DataFrame as a temporary view
df.createOrReplaceTempView("music_logs")

# Use Spark SQL to perform the same count
song_plays_by_user_sql = spark.sql("""
SELECT userId, COUNT(*) AS song_plays
FROM music_logs
WHERE page = 'NextSong'


GROUP BY userId
ORDER BY song_plays DESC
""")

song_plays_by_user_sql.show()

### Assignment:
Analyze user engagement by calculating metrics **per user**: `total songs played`, `total thumbs up given`, and `songs added to playlists`.
- using SparkSQL
- using DataFrames
- using RDD

**Using SparkSQL**

**Using DataFrames**

**Using RDD**